# Churn Prediction Model

In this notebook, we build a customer churn prediction model using the Olist dataset.

Our objective is to predict whether a customer is likely to **churn within the next 90 days**, based on their historical purchase behavior up to a given snapshot date.

This notebook covers:
- building customer-level training snapshots over time,
- defining a churn label,
- engineering predictive customer behavior features,
- training baseline and tree-based classification models,
- evaluating model performance,
- interpreting the most important drivers of churn.

Here, we move from descriptive analytics into **predictive customer analytics**.

We begin by importing the Python libraries needed for data manipulation, visualization, feature engineering, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)

For churn modeling, we need customer order history over time.

We load the main Olist datasets required for this task:
- `orders` for purchase timestamps and order status,
- `customers` to map orders to unique customers,
- `payments` to calculate customer monetary value.

We will use only completed customer activity that is meaningful for retention analysis.

In [ ]:
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")

orders.shape, customers.shape, payments.shape

We convert the purchase timestamp into datetime format so that we can create time-based training snapshots.

We also inspect the purchase date range because churn modeling depends heavily on the observation window available in the dataset.

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

We now create a clean order-level table for modeling.

Steps:
1. keep only delivered orders,
2. join customer identifiers,
3. aggregate payment values to order level,
4. create a normalized purchase date.

This gives us one row per order with the customer and monetary information needed for feature engineering.

In [ ]:
order_payments = (
    payments.groupby("order_id", as_index=False)["payment_value"]
    .sum()
)

model_orders = (
    orders.loc[orders["order_status"] == "delivered", [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_status"
    ]]
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .merge(order_payments, on="order_id", how="left")
    .copy()
)

model_orders["order_date"] = model_orders["order_purchase_timestamp"].dt.normalize()

model_orders = model_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).reset_index(drop=True)

model_orders.head()

Before building a churn model, it is useful to inspect how many customers are repeat buyers.

This is important because Olist is known to have a high proportion of one-time buyers, which makes churn prediction more challenging and also more realistic.

In [ ]:
customer_order_counts = (
    model_orders.groupby("customer_unique_id")["order_id"]
    .nunique()
    .rename("n_orders")
)

customer_order_counts.describe()

To make the model more informative, we calculate the gap in days between consecutive purchases for each customer.

This helps us later derive behavioral features such as:
- average days between orders,
- purchase rhythm,
- repeat purchase consistency.

These features often improve churn prediction beyond basic RFM metrics.

In [ ]:
model_orders["prev_order_timestamp"] = (
    model_orders.groupby("customer_unique_id")["order_purchase_timestamp"]
    .shift(1)
)

model_orders["days_since_prev_order"] = (
    model_orders["order_purchase_timestamp"] - model_orders["prev_order_timestamp"]
).dt.days

model_orders.head()

A churn model should be trained using historical snapshots, not only the final state of each customer.

For each monthly snapshot date:
- we compute customer features using data available **up to that date**,
- then label whether the customer churns in the **next 90 days**.

This creates a proper supervised learning dataset with many customer-date observations.

In [ ]:
CHURN_HORIZON_DAYS = 90

snapshot_dates = pd.date_range(
    start="2017-03-31",
    end="2018-06-30",
    freq="ME"
)

snapshot_dates

The function below builds the training data for a single snapshot date.

For each customer active up to the snapshot date, we calculate:
- frequency features,
- monetary features,
- recency features,
- tenure features,
- inter-purchase timing features.

We then assign the churn label:
- `1` if the customer makes **no purchase** in the next 90 days,
- `0` otherwise.

In [ ]:
def build_snapshot(snapshot_date, orders_df, horizon_days=90):
    history = orders_df.loc[orders_df["order_date"] <= snapshot_date].copy()
    future = orders_df.loc[
        (orders_df["order_date"] > snapshot_date) &
        (orders_df["order_date"] <= snapshot_date + pd.Timedelta(days=horizon_days))
    ].copy()

    if history.empty:
        return pd.DataFrame()

    customer_features = (
        history.groupby("customer_unique_id")
        .agg(
            n_orders=("order_id", "nunique"),
            first_order_date=("order_date", "min"),
            last_order_date=("order_date", "max"),
            total_spend=("payment_value", "sum"),
            avg_order_value=("payment_value", "mean"),
            avg_days_between_orders=("days_since_prev_order", "mean"),
            std_days_between_orders=("days_since_prev_order", "std")
        )
        .reset_index()
    )

    customer_features["snapshot_date"] = snapshot_date
    customer_features["recency_days"] = (
        snapshot_date - customer_features["last_order_date"]
    ).dt.days
    customer_features["tenure_days"] = (
        snapshot_date - customer_features["first_order_date"]
    ).dt.days

    customer_features["orders_per_month"] = np.where(
        customer_features["tenure_days"] > 0,
        customer_features["n_orders"] / (customer_features["tenure_days"] / 30.0),
        customer_features["n_orders"]
    )

    future_customers = set(future["customer_unique_id"].unique())

    customer_features["churned_90d"] = (
        ~customer_features["customer_unique_id"].isin(future_customers)
    ).astype(int)

    return customer_features

We now loop over all snapshot dates and stack the results together into a single modeling table.

Each row represents one customer observed at one point in time, along with:
- historical behavior features,
- and the future churn outcome.

This converts raw transaction history into a machine learning dataset.

In [ ]:
snapshot_frames = []

for snapshot_date in snapshot_dates:
    snapshot_df = build_snapshot(snapshot_date, model_orders, CHURN_HORIZON_DAYS)
    snapshot_frames.append(snapshot_df)

churn_model_df = pd.concat(snapshot_frames, ignore_index=True)

churn_model_df.shape

We inspect the generated churn modeling table to verify the structure and confirm that the expected customer-level features and churn label are present.

In [ ]:
churn_model_df.head()

Class balance is very important in churn prediction.

If churned and non-churned customers are highly imbalanced, some models may appear accurate while performing poorly on the minority class.

So we inspect the churn rate before training.

In [ ]:
churn_model_df["churned_90d"].value_counts(normalize=True).rename("proportion")

A quick visualization helps us see whether the dataset is moderately balanced or heavily skewed toward one class.

In [ ]:
churn_model_df["churned_90d"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["Not churned", "Churned"], rotation=0)
plt.ylabel("Number of snapshot records")
plt.title("Churn class distribution")
plt.show()

Some behavioral features, especially inter-purchase gap features, may be missing for customers with only one order.

This is expected. We will handle these missing values during preprocessing rather than dropping those customers.

In [ ]:
churn_model_df.isna().mean().sort_values(ascending=False)

We now select the numerical features that will be used by the churn model.

These features are intentionally business-interpretable:
- recency,
- frequency,
- spend,
- average order value,
- tenure,
- purchase pace,
- inter-order timing.

In [ ]:
feature_cols = [
    "n_orders",
    "total_spend",
    "avg_order_value",
    "avg_days_between_orders",
    "std_days_between_orders",
    "recency_days",
    "tenure_days",
    "orders_per_month"
]

target_col = "churned_90d"

X = churn_model_df[feature_cols].copy()
y = churn_model_df[target_col].copy()

Because this is a time-based prediction problem, we should not randomly split the data.

Instead, we split by snapshot date:
- earlier snapshots for training,
- mid-period snapshots for validation,
- latest snapshots for testing.

This better reflects how a churn model would behave in production.

In [ ]:
train_mask = churn_model_df["snapshot_date"] <= pd.Timestamp("2017-12-31")
valid_mask = (
    (churn_model_df["snapshot_date"] >= pd.Timestamp("2018-01-31")) &
    (churn_model_df["snapshot_date"] <= pd.Timestamp("2018-03-31"))
)
test_mask = churn_model_df["snapshot_date"] >= pd.Timestamp("2018-04-30")

X_train = churn_model_df.loc[train_mask, feature_cols].copy()
y_train = churn_model_df.loc[train_mask, target_col].copy()

X_valid = churn_model_df.loc[valid_mask, feature_cols].copy()
y_valid = churn_model_df.loc[valid_mask, target_col].copy()

X_test = churn_model_df.loc[test_mask, feature_cols].copy()
y_test = churn_model_df.loc[test_mask, target_col].copy()

X_train.shape, X_valid.shape, X_test.shape

Our model features are all numeric, but some contain missing values.

We create a preprocessing pipeline that:
- imputes missing values with the median,
- scales the features for models that are sensitive to feature magnitude.

This is especially useful for logistic regression.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols)
    ]
)

We start with logistic regression as a baseline model.

This is a strong first benchmark because:
- it is simple,
- easy to interpret,
- and commonly used in business churn modeling.

We also use `class_weight="balanced"` to make the model more robust to class imbalance.

In [ ]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

log_reg_model.fit(X_train, y_train)

We first evaluate the baseline model on the validation set.

We use:
- ROC AUC to measure ranking quality,
- Average Precision to assess performance under class imbalance,
- and a classification report for threshold-based performance.

In [ ]:
valid_pred_log = log_reg_model.predict(X_valid)
valid_proba_log = log_reg_model.predict_proba(X_valid)[:, 1]

print("Logistic Regression - Validation Metrics")
print(f"ROC AUC: {roc_auc_score(y_valid, valid_proba_log):.4f}")
print(f"Average Precision: {average_precision_score(y_valid, valid_proba_log):.4f}")
print()
print(classification_report(y_valid, valid_pred_log))

Next, we train a Random Forest classifier.

This model can capture non-linear relationships and feature interactions that logistic regression may miss, while still remaining interpretable enough for a portfolio project.

In [ ]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), feature_cols)
    ]
)

rf_model = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

We evaluate the Random Forest model on the validation set using the same metrics as before.

This allows us to compare whether the additional model complexity improves churn prediction.

In [ ]:
valid_pred_rf = rf_model.predict(X_valid)
valid_proba_rf = rf_model.predict_proba(X_valid)[:, 1]

print("Random Forest - Validation Metrics")
print(f"ROC AUC: {roc_auc_score(y_valid, valid_proba_rf):.4f}")
print(f"Average Precision: {average_precision_score(y_valid, valid_proba_rf):.4f}")
print()
print(classification_report(y_valid, valid_pred_rf))

We compare the validation performance of both candidate models.

The model with the stronger ranking quality and more useful classification performance will be selected for the final test evaluation.

In [ ]:
comparison = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest"],
    "roc_auc": [
        roc_auc_score(y_valid, valid_proba_log),
        roc_auc_score(y_valid, valid_proba_rf)
    ],
    "avg_precision": [
        average_precision_score(y_valid, valid_proba_log),
        average_precision_score(y_valid, valid_proba_rf)
    ]
})

comparison.sort_values("roc_auc", ascending=False)

Based on the validation metrics, we select the better-performing model and evaluate it on the test set.

This gives us an unbiased estimate of how the churn model performs on the latest unseen customer snapshots.

In [ ]:
best_model_name = comparison.sort_values("roc_auc", ascending=False).iloc[0]["model"]

best_model = rf_model if best_model_name == "Random Forest" else log_reg_model
best_model_name

We now evaluate the selected model on the held-out test set.

This is the most important performance check in the notebook because it reflects generalization to future customer periods.

In [ ]:
test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

print(f"{best_model_name} - Test Metrics")
print(f"ROC AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(f"Average Precision: {average_precision_score(y_test, test_proba):.4f}")
print()
print(classification_report(y_test, test_pred))

The confusion matrix helps us understand how the model behaves in practical churn classification:
- true churners identified correctly,
- loyal customers classified correctly,
- and the trade-off between false positives and false negatives.

In [ ]:
cm = confusion_matrix(y_test, test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not churned", "Churned"])
disp.plot()
plt.title(f"{best_model_name} - Test Confusion Matrix")
plt.show()

The ROC curve shows how well the model separates churners from non-churners across all probability thresholds.

A stronger curve closer to the top-left corner indicates better discriminative performance.

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_proba)
plt.title(f"{best_model_name} - ROC Curve")
plt.show()

For churn problems, the Precision-Recall curve is especially useful because it focuses more directly on the quality of positive class prediction.

This matters when the business goal is to identify high-risk customers for retention campaigns.

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, test_proba)
plt.title(f"{best_model_name} - Precision-Recall Curve")
plt.show()

Model interpretation is important in customer analytics.

If the selected model is Random Forest, we inspect feature importances to understand which customer behavior variables contribute most to churn risk.

In [ ]:
if best_model_name == "Random Forest":
    importances = best_model.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importances
    }).sort_values("importance", ascending=False)

    display(importance_df)

    importance_df.plot(
        kind="barh",
        x="feature",
        y="importance",
        figsize=(8, 5),
        legend=False
    )
    plt.gca().invert_yaxis()
    plt.title("Random Forest Feature Importance")
    plt.xlabel("Importance")
    plt.ylabel("")
    plt.show()

A useful business output is a scored customer table for the most recent snapshot.

We create churn probabilities for the latest snapshot so that the business can identify:
- customers at highest risk,
- retention campaign targets,
- and segments requiring proactive intervention.

In [ ]:
latest_snapshot_date = churn_model_df["snapshot_date"].max()

latest_snapshot = churn_model_df.loc[
    churn_model_df["snapshot_date"] == latest_snapshot_date,
    ["customer_unique_id", "snapshot_date"] + feature_cols + [target_col]
].copy()

latest_snapshot["churn_probability"] = best_model.predict_proba(
    latest_snapshot[feature_cols]
)[:, 1]

latest_snapshot = latest_snapshot.sort_values(
    "churn_probability", ascending=False
).reset_index(drop=True)

latest_snapshot.head(10)

To make the model output more actionable, we convert churn probabilities into simple risk bands.

These bands make it easier for a business team to prioritize retention actions without needing to interpret raw probabilities.

In [ ]:
latest_snapshot["churn_risk_band"] = pd.cut(
    latest_snapshot["churn_probability"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

latest_snapshot["churn_risk_band"].value_counts()

Finally, we save the scored latest customer snapshot for reuse in later notebooks, especially the business recommendations notebook.

This file can be used to design retention actions targeted at high-risk customers.

In [ ]:
latest_snapshot.to_csv(OUTPUT_DIR / "customer_churn_scores.csv", index=False)
churn_model_df.to_csv(OUTPUT_DIR / "churn_modeling_table.csv", index=False)

## Conclusion

In this notebook, we built a churn prediction workflow using the Olist customer transaction history.

Key outcomes:
- we defined churn as no purchase in the next 90 days,
- created monthly customer snapshots for supervised learning,
- engineered interpretable behavioral features,
- trained and compared baseline and tree-based models,
- and produced customer-level churn risk scores for the latest snapshot.

These outputs are valuable because they turn historical purchase behavior into a forward-looking retention signal.

In the next notebook, we can extend customer analytics further by estimating **Customer Lifetime Value (CLV)**, which complements churn probability by quantifying the potential future value of each customer.